In [130]:
from gdt.core import data_path
from gdt.missions.fermi.gbm.tte import GbmTte
from gdt.core.binning.unbinned import bin_by_time
from gdt.core.plot.lightcurve import Lightcurve
from gdt.missions.fermi.time import Time
from scipy import stats
from scipy.optimize import curve_fit
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import pandas
import requests
import json
import os
import traceback
import sys
import statistics
import scipy as sp

In [131]:
with open('data/grb_analysis.json', 'r') as file:
    data = json.load(file)

In [132]:
r = requests.get("https://icecube.wisc.edu/~grbweb_public/GRBweb2.sqlite")
f = open('GRBweb2.sqlite', 'wb').write(r.content)
db = sqlite3.connect('GRBweb2.sqlite')
save_array = {}

In [133]:
names = [grb for grb in data]

In [134]:
names[9]
data[names[9]]["low_t"]

382584705.523908

In [135]:
H_naught_mpc = 67.66
H_err = 0.42
Lambda_m = 0.3103
m_err = 0.0057
Lambda_G = 0.6897
G_err = 0.0057
Lambda_k = 0.685
conversion= 3.08567758128*10**19
H_naught = H_naught_mpc/conversion

In [136]:
def f(t):
    num = np.float128((1+t))
    den = np.float128((Lambda_m*(1+t)**3)+Lambda_G)
    return num/(np.sqrt(den))

In [137]:
data["120729456"]["low_t"]

0

In [190]:
def lat_query_by_name(name):
    name = "GRB" + name
    #print("Checking: " + name)
    Sum_table = pandas.read_sql_query("SELECT * from Summary", db)
    Sum_table = Sum_table.sort_values("GRB_name_Fermi")
    indices = np.where(Sum_table == name)
    row_indices, col_indices = indices[0], indices[1]
    index = Sum_table.iloc[row_indices].to_numpy()[0][0] - 1
    
    low_t = data[name]["low_t"]

    E = 300 #300
    z = float(data[name]["redshift"])
    K_int = sp.integrate.quad(f, 0, z)[0]
    K = float(1/(H_naught))*float((E)/(1+z))*float(K_int)
    E=(K/(3*10**17))

    time_window = E *(1+float(data[name]["redshift"]))
    #print(str(time_window))
    low_check = low_t - time_window
    high_check = low_t + time_window
    fermi_met = Time(low_t, format='fermi')
    fermi_low = Time(low_check, format='fermi')
    fermi_high = Time(high_check, format='fermi')

    #print(str(names[ind]) + "-")
    
    #print(str(ind+1) + " - " + str(names[ind]))
    print("RA,DEC:")
    print(str(Sum_table.iloc[row_indices].ra.get(index))+","+str(Sum_table.iloc[row_indices].decl.get(index)))
    print("Time (MJD):")
    print(str(fermi_low.mjd)+","+str(fermi_high.mjd))
    print(str(time_window))
    print("=======================================================")
    
    
    #Sum_table.iloc[row_indices].T100.get(index)
    #Sum_table.iloc[row_indices].T0.get(index)
    #Sum_table.iloc[row_indices].GRB_name.get(index)
    Sum_table.iloc[row_indices]

In [141]:
def lat_query(ind):
    name = "GRB" + names[ind]
    #print("Checking: " + name)
    Sum_table = pandas.read_sql_query("SELECT * from Summary", db)
    Sum_table = Sum_table.sort_values("GRB_name_Fermi")
    indices = np.where(Sum_table == name)
    row_indices, col_indices = indices[0], indices[1]
    index = Sum_table.iloc[row_indices].to_numpy()[0][0] - 1
    
    low_t = data[names[ind]]["low_t"]

    E = 300 #300
    z = float(data[names[ind]]["redshift"])
    K_int = sp.integrate.quad(f, 0, z)[0]
    K = float(1/(H_naught))*float((E)/(1+z))*float(K_int)
    E=(K/(3*10**17))

    time_window = E *(1+float(data[names[ind]]["redshift"]))
    #print(str(time_window))
    low_check = low_t - time_window
    high_check = low_t + time_window
    fermi_met = Time(low_t, format='fermi')
    fermi_low = Time(low_check, format='fermi')
    fermi_high = Time(high_check, format='fermi')

    print(str(names[ind]) + "-")
    
    #print(str(ind+1) + " - " + str(names[ind]))
    print("RA,DEC:")
    print(str(Sum_table.iloc[row_indices].ra.get(index))+","+str(Sum_table.iloc[row_indices].decl.get(index)))
    print("Time (MJD):")
    print(str(fermi_low.mjd)+","+str(fermi_high.mjd))
    print(str(time_window))
    print("=======================================================")
    
    
    #Sum_table.iloc[row_indices].T100.get(index)
    #Sum_table.iloc[row_indices].T0.get(index)
    #Sum_table.iloc[row_indices].GRB_name.get(index)
    Sum_table.iloc[row_indices]

In [143]:
lat_query(6)

150727793-
7 - 150727793
RA,DEC:
203.96883333333332,-18.325333333333333
Time (MJD):
57230.79347227602,57230.79699141843
152.02695199431002


In [145]:
missed_list = ["260208214", "260131277", "260114486", "260101039", "251222712", "251214377", "251201685", "230812790", "200411187", "200219317", "180805543", "180727594", "180618030", "180418281", "170903534", "170817529", "170728961", "170127634", "160411062", "160408268", "151229285", "140713780", "131229277", "130907904", "130716442", "130515056", "121123421", "110402009", "101224227", "81008832", "80916009", "80810549", "80804972"]

In [191]:
ind = 33
print(missed_list[ind-1])
print("==========")
lat_query_by_name(missed_list[ind-1])

80804972


IndexError: index 0 is out of bounds for axis 0 with size 0

In [106]:
for i in range(len(data)):
    lat_query(i)

210606164-
111228657-
090902462-
241025067-
111117510-
091208410-
150727793-
091024372-
250226274-
130215063-
191011192-
100625773-
090113778-
160104475-
210323918-
120729456-
161017745-
250919020-
250725077-
100728439-
140907672-
150821406-
170214649-
160623209-
231118720-
150314205-
250920366-
140506880-
100728095-
140808038-
201020732-
251103199-
251001595-
160629930-
081222204-
250316375-
080916406-
131004904-
121011469-
120624933-
211018936-
100814160-
160625945-
200817393-
141220252-
210722871-
240218084-
161129300-
230328621-
251003082-
240619155-
190114873-
140206304-
240514169-
250617876-
131231198-
121128212-
090927422-
251202077-
180314030-
130610133-
090323002-
210510806-
180620660-
140801792-
151111356-
121217313-
150101641-
181020792-
150403913-
151027166-
140108721-
240912074-
150301818-
100615083-
090510016-
210826293-
120909070-
250916562-
110731465-
250225819-
201221963-
200826187-
140423356-
090926181-
240825662-
091003191-
241209233-
240511754-
100413732-
090328401-

In [93]:
for i in range(250):
    lat_query(i)

Checking: GRB210606164

1 - 210606164
RA,DEC:
170.94108333333335,0.8129166666666666
Time (MJD):
59369.28363359075,59373.05677860652
Checking: GRB111228657

2 - 111228657
RA,DEC:
150.06670833333334,18.297722222222223
Time (MJD):
55923.24121657537,55924.073702099486
Checking: GRB090902462

3 - 090902462
RA,DEC:
264.93921,27.324194
Time (MJD):
55075.40966409849,55077.515891381
Checking: GRB241025067

4 - 241025067
RA,DEC:
333.6537916666667,83.57563888888889
Time (MJD):
60605.90093425951,60610.235757269336
Checking: GRB111117510

5 - 111117510
RA,DEC:
12.704166666666666,23.016666666666666
Time (MJD):
55881.253006521,55883.767553970756
Checking: GRB091208410

6 - 091208410
RA,DEC:
29.392208,16.889722
Time (MJD):
55172.785861848366,55174.03526407733
Checking: GRB150727793

7 - 150727793
RA,DEC:
203.96883333333332,-18.325333333333333
Time (MJD):
57230.61927472686,57230.97118896759
Checking: GRB091024372

8 - 091024372
RA,DEC:
339.24,56.885
Time (MJD):
55127.73123158355,55129.01476740034
Check

IndexError: list index out of range